_____________________________________________________________________________________________________________________________________

### PROYEK AKHIR PRAKTIKUM BIG DATA

## “TidarMart Data Platform: Analitik Big Data End-to-End pada Resource Terbatas”

------------------------------------------------------------------------------------

## Nama : Albani Hafiz Innopha
## NPM : 2505060001
## Varian : 1 (Fokus nilai pelanggan)

_____________________________________________________________________________________________________________________________________

## Tahap 1 - Penyimpanan Terdistribusi dan Ingesti (HDFS)
### Tujuan
1. Membangun struktur direktori *Data Lake* di HDFS untuk memisahkan zona data mentah (`raw`), zona *staging* dan input aliran (`stream_staging`, `stream_input`), zona data bersih (`curated`), serta penyimpanan model (`models`).
2. Mengunggah seluruh dataset multi-format dari lokal ke HDFS agar seluruh tahapan pemrosesan membaca langsung dari `hdfs://localhost:9000/tidarmart/...`.
3. Membuktikan karakteristik **Volume** melalui pemeriksaan kapasitas penyimpanan, jumlah blok HDFS, faktor replikasi, dan inisialisasi `SparkSession` yang dioptimalkan untuk VM dengan RAM terbatas.

In [2]:
#Membuat direktori yang diperluakan di HDFS
!hdfs dfs -mkdir -p /tidarmart/raw/transaksi /tidarmart/raw/pelanggan /tidarmart/raw/produk
!hdfs dfs -mkdir -p /tidarmart/stream_staging /tidarmart/stream_input
!hdfs dfs -mkdir -p /tidarmart/curated /tidarmart/models
!hdfs dfs -ls -R /tidarmart

#Mengunggah data lokal ke HDFS
!hdfs dfs -put -f data_proyek/transaksi/*.csv /tidarmart/raw/transaksi/
!hdfs dfs -put -f data_proyek/pelanggan.json /tidarmart/raw/pelanggan/
!hdfs dfs -put -f data_proyek/produk.json /tidarmart/raw/produk/
!hdfs dfs -put -f data_proyek/stream_input/*.csv /tidarmart/stream_staging/

drwxr-xr-x   - albani supergroup          0 2026-11-26 07:59 /tidarmart/curated
drwxr-xr-x   - albani supergroup          0 2026-11-26 07:59 /tidarmart/models
drwxr-xr-x   - albani supergroup          0 2026-11-26 08:13 /tidarmart/raw
drwxr-xr-x   - albani supergroup          0 2026-11-26 08:03 /tidarmart/raw/pelanggan
-rw-r--r--   1 albani supergroup    6307827 2026-11-26 08:03 /tidarmart/raw/pelanggan/pelanggan.json
drwxr-xr-x   - albani supergroup          0 2026-11-26 08:04 /tidarmart/raw/produk
-rw-r--r--   1 albani supergroup     700972 2026-11-26 08:04 /tidarmart/raw/produk/produk.json
drwxr-xr-x   - albani supergroup          0 2026-11-26 08:02 /tidarmart/raw/transaksi
-rw-r--r--   1 albani supergroup   49066043 2026-11-26 08:02 /tidarmart/raw/transaksi/transaksi_part1.csv
-rw-r--r--   1 albani supergroup   49064207 2026-11-26 08:02 /tidarmart/raw/transaksi/transaksi_part2.csv
-rw-r--r--   1 albani supergroup   49061114 2026-11-26 08:02 /tidarmart/raw/transaksi/transaksi_part3.

penjelasan: 20 berkas stream disimpan di /tidarmart/stream_staging, sedangkan /tidarmart/stream_input sengaja kosong. Di Tahap 6 berkas disalin satu per satu ke stream_input untuk mensimulasikan data yang masuk terus-menerus dan membuktikan hasil streaming bertambah.

In [3]:
#Bukti Volume: ukuran, jumlah blok, faktor replikasi
!hdfs dfs -du -h /tidarmart/raw
!hdfs dfs -du -s -h /tidarmart/raw/transaksi
!hdfs fsck /tidarmart/raw/transaksi -files -blocks
!hdfs dfs -stat "%n  replikasi=%r  blocksize=%o" /tidarmart/raw/transaksi/transaksi_part1.csv

6.0 M    6.0 M    /tidarmart/raw/pelanggan
684.5 K  684.5 K  /tidarmart/raw/produk
187.2 M  187.2 M  /tidarmart/raw/transaksi
187.2 M  187.2 M  /tidarmart/raw/transaksi
Connecting to namenode via http://localhost:9870/fsck?ugi=albani&files=1&blocks=1&path=%2Ftidarmart%2Fraw%2Ftransaksi
FSCK started by albani (auth:SIMPLE) from /127.0.0.1 for path /tidarmart/raw/transaksi at Thu Nov 26 23:14:44 WIB 2026

/tidarmart/raw/transaksi <dir>
/tidarmart/raw/transaksi/transaksi_part1.csv 49066043 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-685778691-127.0.1.1-1789562457416:blk_1073741903_1079 len=49066043 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part2.csv 49064207 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-685778691-127.0.1.1-1789562457416:blk_1073741904_1080 len=49064207 Live_repl=1

/tidarmart/raw/transaksi/transaksi_part3.csv 49061114 bytes, replicated: replication=1, 1 block(s):  OK
0. BP-685778691-127.0.1.1-1789562457416:blk_1073741905_1081 len=49061114 Live_

In [1]:
#Inisialisasi SparkSession
from pyspark.sql import SparkSession
spark = (SparkSession.builder
    .appName('TidarMart')
    .master('local[2]')
    .config('spark.driver.memory', '1536m')
    .config('spark.sql.shuffle.partitions', '4')
    .config('spark.sql.adaptive.enabled', 'true')
    .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

HDFS = "hdfs://localhost:9000/tidarmart"
df = spark.read.csv(f"{HDFS}/raw/transaksi/*.csv", header=True, inferSchema=True)
print(f"{df.count():,} baris")
df.printSchema()

26/11/26 23:13:48 WARN Utils: Your hostname, albani resolves to a loopback address: 127.0.1.1; using 192.168.18.133 instead (on interface wlp0s20f3)
26/11/26 23:13:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/26 23:13:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
                                                                                

2,010,000 baris
root
 |-- order_id: string (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- pelanggan_id: integer (nullable = true)
 |-- produk_id: integer (nullable = true)
 |-- qty: double (nullable = true)
 |-- harga_satuan: double (nullable = true)
 |-- diskon_persen: integer (nullable = true)
 |-- metode_pembayaran: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- berat_kg: double (nullable = true)
 |-- jarak_km: double (nullable = true)
 |-- lama_kirim_hari: integer (nullable = true)
 |-- status_pesanan: string (nullable = true)



### Interpretasi & Poin Presentasi Tahap 1

1. **Mengapa direktori `raw` dan `curated` dipisah?**
   * Zona `raw` berfungsi sebagai *single source of truth* yang bersifat *immutable* (tidak pernah ditimpa atau diubah). Jika di kemudian hari aturan pembersihan data perlu direvisi, proses ETL dapat dijalankan ulang dari `raw` tanpa kehilangan data asli. Sementara itu, zona `curated` menyimpan data bersih berformat kolumnar (Parquet) yang siap dipakai cepat oleh tim analitik dan *Machine Learning*.
2. **Makna Faktor Replikasi = 1 pada Lingkungan *Pseudo-Distributed* vs. Klaster Nyata:**
   * Pada VM VirtualBox (*pseudo-distributed*), hanya terdapat 1 `DataNode`, sehingga `replication = 1` sudah cukup untuk simulasi dan menghemat ruang disk, namun belum memiliki toleransi kegagalan perangkat keras. Pada klaster produksi nyata dengan banyak server, replikasi umumnya diatur menjadi **3** dan disebar ke rak berbeda agar data tetap aman meski ada node yang mati.
3. **Trade-off `inferSchema=True` vs. Skema Eksplisit (`StructType`):**
   * Menggunakan `inferSchema=True` memaksa Spark membaca dataset **dua kali** (membaca sampel/keseluruhan data untuk menebak tipe kolom, lalu membacanya lagi untuk memuat data) dan berisiko salah menebak tipe jika isi data berubah. Karena itu, pada tahap berikutnya kita menggunakan skema eksplisit agar lebih cepat dan deterministik.
4. **Bukti Karakteristik Volume:** *(Catat angka dari output Sel 1.3 & 1.4: 4 berkas CSV transaksi berukuran ~187 MB / ~2.010.000 baris, masing-masing menempati 1 blok HDFS karena ukurannya < 128 MB block size).*

## Tahap 2 — Profiling Kualitas Data (*Veracity & Variety*)

### Tujuan
1. Mendeteksi dan mengukur seluruh masalah kualitas data (*Veracity*) pada data transaksi mentah **sebelum** dilakukan pembersihan: duplikat `order_id`, nilai kosong (`NULL`/`NaN`), harga satuan negatif, *outlier* kuantitas berbasis kuantil (IQR), dan ketidakkonsistenan penulisan nama kota.
2. Menyusun tabel ringkasan **Sebelum Pembersihan** dalam bentuk DataFrame PySpark murni sebagai patokan (*baseline*) aturan ETL.
3. Memeriksa struktur semi-terstruktur pada `pelanggan.json` dan `produk.json` menggunakan skema eksplisit untuk membuktikan karakteristik **Variety** (atribut berbeda per kategori produk dan penanganan array opsional `tags`).

In [4]:
# LANGKAH 2.1: Audit nilai kosong (NULL/NaN) dan anomali angka dalam SATU kali scan (Single-Pass)
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType

# Fungsi bantu untuk mendeteksi NULL sekaligus NaN pada kolom desimal
def cek_kosong(df_in, col_name):
    tipe = df_in.schema[col_name].dataType
    if isinstance(tipe, (DoubleType, FloatType)):
        return F.col(col_name).isNull() | F.isnan(F.col(col_name))
    return F.col(col_name).isNull()

ekspr_audit = [
    F.count("*").alias("total_baris"),
    F.sum(F.when(F.col("harga_satuan") < 0, 1).otherwise(0)).alias("harga_negatif"),
    F.sum(F.when(F.col("qty") > 7, 1).otherwise(0)).alias("qty_gt_7"),
    F.sum(F.when(F.col("qty") == 999, 1).otherwise(0)).alias("qty_999"),
    F.min("tanggal").alias("tgl_awal"),
    F.max("tanggal").alias("tgl_akhir"),
] + [
    F.sum(F.when(cek_kosong(df, c), 1).otherwise(0)).alias(f"null_{c}")
    for c in df.columns
]

# .first().asDict() hanya menarik 1 baris hasil ringkasan ke driver (aman dari OOM)
r = df.agg(*ekspr_audit).first().asDict()
total = r["total_baris"]

print(f"Total baris transaksi : {total:,}")
print(f"Rentang tanggal       : {r['tgl_awal']} s/d {r['tgl_akhir']}")
print(f"Harga satuan negatif  : {r['harga_negatif']:,} baris")
print(f"Kuantitas > 7         : {r['qty_gt_7']:,} baris | qty == 999: {r['qty_999']:,} baris")
print("\nKolom yang memiliki nilai kosong / NaN:")
for c in df.columns:
    n_null = r[f"null_{c}"]
    if n_null > 0:
        print(f"  - {c:16s}: {n_null:,} baris ({n_null / total * 100:.3f}%)")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


[Stage 5:>                                                          (0 + 2) / 2]

Total baris transaksi : 2,010,000
Rentang tanggal       : 2026-01-01 00:00:03 s/d 2026-06-30 23:59:50
Harga satuan negatif  : 2,082 baris
Kuantitas > 7         : 961 baris | qty == 999: 961 baris

Kolom yang memiliki nilai kosong / NaN:
  - qty             : 6,122 baris (0.305%)
  - harga_satuan    : 6,215 baris (0.309%)


In [5]:
# LANGKAH 2.2: Pemeriksaan baris duplikat (order_id vs seluruh kolom)
n_unik_id   = df.select("order_id").distinct().count()
n_dedup_all = df.dropDuplicates().count()
n_dedup_id  = df.dropDuplicates(["order_id"]).count()
n_duplikat  = total - n_unik_id

print(f"Jumlah order_id unik          : {n_unik_id:,}")
print(f"Jumlah baris duplikat         : {n_duplikat:,} ({n_duplikat / total * 100:.3f}%)")
print(f"Sisa baris dropDuplicates()   : {n_dedup_all:,}")
print(f"Sisa baris dropDuplicates(id) : {n_dedup_id:,}")
print(f"Apakah duplikat 100% identik? : {n_dedup_all == n_dedup_id}")

[Stage 22:>                                                         (0 + 2) / 2]

Jumlah order_id unik          : 2,000,000
Jumlah baris duplikat         : 10,000 (0.498%)
Sisa baris dropDuplicates()   : 2,000,000
Sisa baris dropDuplicates(id) : 2,000,000
Apakah duplikat 100% identik? : True


In [6]:
# LANGKAH 2.3: Deteksi outlier kuantitas (qty) secara statistik menggunakan Kuantil & IQR
# Saring nilai valid terlebih dahulu agar NaN tidak memengaruhi perhitungan kuantil
q = df.filter(~cek_kosong(df, "qty")).approxQuantile("qty", [0.01, 0.25, 0.5, 0.75, 0.99, 0.999, 1.0], 0.001)
iqr_qty = q[3] - q[1]
batas_qty = q[3] + 1.5 * iqr_qty  # Rumus Tukey: Q3 + 1.5 * IQR

print("Ringkasan Kuantil qty:", dict(zip(["p1", "Q1(p25)", "Median", "Q3(p75)", "p99", "p99.9", "max"], q)))
print(f"IQR = {iqr_qty} | Batas atas outlier (Q3 + 1.5*IQR) = {batas_qty}")
df.groupBy("qty").count().orderBy("qty").show(15)

Ringkasan Kuantil qty: {'p1': 1.0, 'Q1(p25)': 2.0, 'Median': 4.0, 'Q3(p75)': 6.0, 'p99': 7.0, 'p99.9': 999.0, 'max': 999.0}
IQR = 4.0 | Batas atas outlier (Q3 + 1.5*IQR) = 12.0


[Stage 27:>                                                         (0 + 2) / 2]

+-----+------+
|  qty| count|
+-----+------+
| NULL|  6122|
|  1.0|285922|
|  2.0|287054|
|  3.0|285062|
|  4.0|286098|
|  5.0|285747|
|  6.0|286171|
|  7.0|286863|
|999.0|   961|
+-----+------+



In [7]:
# LANGKAH 2.4: Pemeriksaan ketidakkonsistenan penulisan nama kota
df.groupBy("kota").count().orderBy("kota").show(30, truncate=False)
n_varian_kota = df.select("kota").distinct().count()
n_kota_norm   = df.select(F.initcap(F.lower(F.trim("kota")))).distinct().count()
n_kota_rusak  = df.filter(F.col("kota") != F.initcap(F.lower(F.trim("kota")))).count()

print(f"Jumlah varian string kota mentah : {n_varian_kota}")
print(f"Jumlah kota setelah normalisasi  : {n_kota_norm}")
print(f"Baris kota tidak konsisten       : {n_kota_rusak:,} ({n_kota_rusak / total * 100:.3f}%)")

+----------+------+
|kota      |count |
+----------+------+
|KEBUMEN   |6378  |
|Kebumen   |238461|
|MAGELANG  |6309  |
|Magelang  |238568|
|PURWOREJO |6272  |
|Purworejo |238396|
|SEMARANG  |6253  |
|SOLO      |6275  |
|Semarang  |238812|
|Solo      |238491|
|TEMANGGUNG|6426  |
|Temanggung|239133|
|WONOSOBO  |6345  |
|Wonosobo  |237884|
|YOGYAKARTA|6333  |
|Yogyakarta|239013|
|kebumen   |6411  |
|magelang  |6331  |
|purworejo |6507  |
|semarang  |6313  |
|solo      |6254  |
|temanggung|6241  |
|wonosobo  |6397  |
|yogyakarta|6197  |
+----------+------+



[Stage 45:>                                                         (0 + 2) / 2]

Jumlah varian string kota mentah : 24
Jumlah kota setelah normalisasi  : 8
Baris kota tidak konsisten       : 101,242 (5.037%)


In [8]:
# LANGKAH 2.5: Profiling VARIETY pada data semi-terstruktur (pelanggan.json & produk.json)
from pyspark.sql.types import (
    StructType, StructField, LongType, StringType,
    DoubleType, IntegerType, ArrayType
)

# Menggunakan skema eksplisit (StructType) agar pembacaan lebih cepat & bebas salah tipe
skema_pel = StructType([
    StructField("pelanggan_id", LongType(), True),
    StructField("nama", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("tanggal_daftar", StringType(), True),
    StructField("membership", StringType(), True)
])

skema_prod = StructType([
    StructField("produk_id", LongType(), True),
    StructField("nama_produk", StringType(), True),
    StructField("kategori", StringType(), True),
    StructField("harga_dasar", LongType(), True),
    StructField("berat_kg", DoubleType(), True),
    StructField("garansi_bulan", IntegerType(), True),
    StructField("voltase", IntegerType(), True),
    StructField("ukuran", StringType(), True),
    StructField("bahan", StringType(), True),
    StructField("kadaluarsa_hari", IntegerType(), True),
    StructField("penerbit", StringType(), True),
    StructField("tags", ArrayType(StringType()), True)
])

pel  = spark.read.schema(skema_pel).json(f"{HDFS}/raw/pelanggan/pelanggan.json")
prod = spark.read.schema(skema_prod).json(f"{HDFS}/raw/produk/produk.json")

n_pel, n_prod = pel.count(), prod.count()
print(f"Total pelanggan: {n_pel:,} | Total produk: {n_prod:,}")
pel.groupBy("membership").count().orderBy(F.desc("count")).show()

# Membuktikan karakteristik Variety: atribut produk berbeda per kategori
atribut_opsional = ["garansi_bulan", "voltase", "ukuran", "bahan", "kadaluarsa_hari", "penerbit", "tags"]
print("Persentase produk yang memiliki tiap atribut opsional (%):")
prod.agg(*[F.round(F.count(c) / n_prod * 100, 1).alias(c) for c in atribut_opsional]).show()

print("Kelengkapan atribut opsional per kategori produk:")
prod.groupBy("kategori").agg(
    F.count("*").alias("jumlah_produk"),
    *[F.count(c).alias(c) for c in atribut_opsional]
).orderBy("kategori").show(truncate=False)

# Penanganan kolom array opsional 'tags' (coalesce ke array kosong & explode)
print("Contoh penanganan array opsional 'tags' & distribusi isi tag:")
(prod.select(
        "produk_id", "kategori", "tags",
        F.coalesce(F.col("tags"), F.array().cast(ArrayType(StringType()))).alias("tags_aman"),
        F.size(F.coalesce(F.col("tags"), F.array().cast(ArrayType(StringType())))).alias("jumlah_tag")
    )
    .show(6, truncate=False))

prod.select(F.explode("tags").alias("tag")).groupBy("tag").count().orderBy(F.desc("count")).show()

Total pelanggan: 50,000 | Total produk: 5,000
+----------+-----+
|membership|count|
+----------+-----+
|    Silver|30114|
|      Gold|14994|
|  Platinum| 4892|
+----------+-----+

Persentase produk yang memiliki tiap atribut opsional (%):
+-------------+-------+------+-----+---------------+--------+----+
|garansi_bulan|voltase|ukuran|bahan|kadaluarsa_hari|penerbit|tags|
+-------------+-------+------+-----+---------------+--------+----+
|         12.5|   12.5|  13.1| 13.1|           13.3|    12.5|30.2|
+-------------+-------+------+-----+---------------+--------+----+

Kelengkapan atribut opsional per kategori produk:
+------------+-------------+-------------+-------+------+-----+---------------+--------+----+
|kategori    |jumlah_produk|garansi_bulan|voltase|ukuran|bahan|kadaluarsa_hari|penerbit|tags|
+------------+-------------+-------------+-------+------+-----+---------------+--------+----+
|Buku        |623          |0            |0      |0     |0    |0              |623     |176 |

In [9]:
# LANGKAH 2.6: Tabel ringkasan SEBELUM pembersihan (100% Spark DataFrame, murni Spark)
baris_sebelum = [
    (1, "Total baris",                f"{total:,}"),
    (2, "order_id unik",              f"{n_unik_id:,}"),
    (3, "Baris duplikat",             f"{n_duplikat:,}"),
    (4, "qty kosong/NaN (%)",         f"{r['null_qty'] / total * 100:.3f}%"),
    (5, "harga_satuan kosong (%)",    f"{r['null_harga_satuan'] / total * 100:.3f}%"),
    (6, "harga_satuan negatif (<0)",  f"{r['harga_negatif']:,}"),
    (7, "Outlier qty (> batas IQR)",  f"{r['qty_gt_7']:,}"),
    (8, "Kota tidak konsisten",       f"{n_kota_rusak:,}"),
    (9, "Varian penulisan kota",      f"{n_varian_kota}"),
]

profil_sebelum = spark.createDataFrame(baris_sebelum, ["no", "Metrik", "Sebelum"])
profil_sebelum.orderBy("no").select("Metrik", "Sebelum").show(truncate=False)

+-------------------------+---------+
|Metrik                   |Sebelum  |
+-------------------------+---------+
|Total baris              |2,010,000|
|order_id unik            |2,000,000|
|Baris duplikat           |10,000   |
|qty kosong/NaN (%)       |0.305%   |
|harga_satuan kosong (%)  |0.309%   |
|harga_satuan negatif (<0)|2,082    |
|Outlier qty (> batas IQR)|961      |
|Kota tidak konsisten     |101,242  |
|Varian penulisan kota    |24       |
+-------------------------+---------+



### Interpretasi & Poin Presentasi Tahap 2

1. **Temuan Kualitas Data (*Veracity*):**
   * Dari **2.010.000 baris** mentah, terdapat **10.000 baris duplikat (0,498%)** dengan isi seluruh kolom 100% identik; jika dibiarkan, omzet dan jumlah transaksi akan terhitung ganda (*double counting*).
   * Nilai kosong (`NULL`/`NaN`) hanya muncul pada `qty` (~0,31%) dan `harga_satuan` (~0,30%), sedangkan kolom lainnya lengkap. Selain itu ditemukan harga satuan negatif (`< 0`) dan lonjakan kuantitas ekstrem (`qty = 999`) jauh di atas distribusi normal 1–7 unit ($Q_1 = 2$, Median $= 4$, $Q_3 = 6$).
   * Inkonsistensi huruf besar/kecil pada kolom `kota` memecah **8 kota operasional** menjadi **24 string kota berbeda**, sehingga agregasi per kota akan terpecah menjadi tiga grup per kota jika tidak dinormalisasi.
2. **Temuan Variasi Skema (*Variety*):**
   * Dataset `produk.json` bersifat semi-terstruktur: atribut spesifik hanya muncul pada kategori relevan (`garansi_bulan` dan `voltase` khusus **Elektronik**, `ukuran` dan `bahan` khusus **Fashion**, `kadaluarsa_hari` khusus **Makanan**, `penerbit` khusus **Buku**), sementara array opsional `tags` hanya terisi pada ~30% produk dan diamankan menggunakan `coalesce(..., array())`.

## Tahap 3 — ETL dan Penyimpanan Kolumnar (Parquet)

### Tujuan
1. Membersihkan dataset transaksi berdasarkan temuan audit Tahap 2 disertai validasi matematis untuk setiap aturan pemulihan data.
2. Menambahkan empat kolom turunan bisnis wajib: `total_nilai`, `bulan`, `hari_dalam_minggu`, dan `flag_batal`.
3. Menyimpan data bersih ke `/tidarmart/curated/transaksi` dalam format **Parquet** berpartisi kolom `bulan`.
4. Memverifikasi ketuntasan pembersihan lewat tabel perbandingan **Sebelum vs. Sesudah Pembersihan** serta membandingkan ukuran disk dan kecepatan query antara **CSV mentah vs. Parquet**.

### Ringkasan Aturan Pembersihan
| # | Masalah Data | Aturan ETL | Justifikasi Teknis |
|---|---|---|---|
| 1 | Duplikat `order_id` | `dropDuplicates(["order_id"])` | Baris duplikat identik di seluruh kolom; dibuang agar omzet tidak terhitung ganda. |
| 2 | `harga_satuan` kosong/NaN | Dipulihkan dari `harga_dasar` produk | Harga satuan standar tercatat di katalog `produk.json`, lebih akurat daripada imputasi median. |
| 3 | `harga_satuan` negatif (`<= 0`) | Dikoreksi ke `harga_dasar` produk | Harga negatif mustahil secara bisnis; nilai asli dapat direkonstruksi dari `produk_id`. |
| 4 | `qty` kosong/NaN | Dihitung ulang: `round(berat_kg / berat_ref)` | Berat total pengiriman (`berat_kg`) merupakan kelipatan berat satuan produk (`berat_ref`). |
| 5 | `qty` outlier (`> batas_qty`) | Dikoreksi dengan rumus `round(berat_kg / berat_ref)` | Ambang ditentukan secara statistik ($Q_3 + 1.5 \times \text{IQR}$); memulihkan `qty` menjaga pesanan sah tetap terhitung. |
| 6 | Penulisan `kota` tidak seragam | `initcap(lower(trim(kota)))` | Hanya perbedaan spasi dan huruf kapital; diseragamkan menjadi 8 nama kota baku. |

In [10]:
# LANGKAH 3.1: Validasi aturan pemulihan harga & qty pada baris normal MAUPUN baris bermasalah
prod_ref = prod.select(
    "produk_id",
    F.col("harga_dasar").alias("harga_ref"),
    F.col("berat_kg").alias("berat_ref")
)

df_join_cek = df.join(F.broadcast(prod_ref), "produk_id", "left")

# 1) Cek kesesuaian rumus pada baris normal
cek_normal = (df_join_cek
    .filter((F.col("harga_satuan") > 0) & (~cek_kosong(df, "qty")) & (F.col("qty") <= batas_qty))
    .agg(
        F.count("*").alias("n_baris_normal"),
        F.round(F.avg((F.col("harga_satuan") == F.col("harga_ref")).cast("int")), 4).alias("pct_harga_cocok"),
        F.round(F.avg((F.round(F.col("berat_kg") / F.col("berat_ref")) == F.col("qty")).cast("int")), 4).alias("pct_qty_cocok")
    ))
print("Validasi rumus pemulihan pada baris normal (target ~1.0):")
cek_normal.show()

# 2) Cek khusus pada baris dengan qty kosong/outlier untuk memastikan berat_kg masih menyimpan berat asli
cek_rusak = (df_join_cek
    .filter(cek_kosong(df, "qty") | (F.col("qty") > batas_qty))
    .withColumn("qty_pulih", F.round(F.col("berat_kg") / F.col("berat_ref")))
    .agg(
        F.count("*").alias("n_qty_bermasalah"),
        F.sum(F.when(F.col("qty_pulih").isNull() | (F.col("qty_pulih") < 1) | (F.col("qty_pulih") > batas_qty), 1).otherwise(0)).alias("gagal_pulih"),
        F.min("qty_pulih").alias("min_qty_pulih"),
        F.max("qty_pulih").alias("max_qty_pulih")
    ))
print("Validasi pemulihan khusus pada baris qty bermasalah (gagal_pulih harus 0):")
cek_rusak.show()

Validasi rumus pemulihan pada baris normal (target ~1.0):


+--------------+---------------+-------------+
|n_baris_normal|pct_harga_cocok|pct_qty_cocok|
+--------------+---------------+-------------+
|       1994654|            1.0|          1.0|
+--------------+---------------+-------------+

Validasi pemulihan khusus pada baris qty bermasalah (gagal_pulih harus 0):


[Stage 73:>                                                         (0 + 2) / 2]

+----------------+-----------+-------------+-------------+
|n_qty_bermasalah|gagal_pulih|min_qty_pulih|max_qty_pulih|
+----------------+-----------+-------------+-------------+
|            7083|          0|          1.0|          7.0|
+----------------+-----------+-------------+-------------+



In [11]:
# LANGKAH 3.2: Eksekusi Pipeline Pembersihan & Penambahan 4 Kolom Turunan Wajib
kondisi_harga_rusak = cek_kosong(df, "harga_satuan") | (F.col("harga_satuan") <= 0)
kondisi_qty_rusak   = cek_kosong(df, "qty") | (F.col("qty") <= 0) | (F.col("qty") > batas_qty)

bersih = (df
    # Aturan 1: Hapus duplikat berdasarkan order_id
    .dropDuplicates(["order_id"])
    # Broadcast join tabel referensi produk (5.000 baris) agar bebas shuffle besar
    .join(F.broadcast(prod_ref), "produk_id", "left")
    # Simpan kolom penanda jejak audit sebelum kolom diperbaiki
    .withColumn("harga_dikoreksi", kondisi_harga_rusak)
    .withColumn("qty_dikoreksi",   kondisi_qty_rusak)
    .withColumn("kota_diubah",     F.col("kota") != F.initcap(F.lower(F.trim("kota"))))
    # Aturan 2 & 3: Pulihkan harga_satuan kosong/negatif dari katalog produk
    .withColumn("harga_satuan",
        F.when(F.col("harga_dikoreksi"), F.col("harga_ref"))
         .otherwise(F.col("harga_satuan")).cast("double"))
    # Aturan 4 & 5: Pulihkan qty kosong/outlier dari rasio berat_kg / berat_ref
    .withColumn("qty",
        F.when(F.col("qty_dikoreksi"), F.round(F.col("berat_kg") / F.col("berat_ref")))
         .otherwise(F.col("qty")).cast("int"))
    # Aturan 6: Normalisasi nama kota
    .withColumn("kota", F.initcap(F.lower(F.trim("kota"))))
    .withColumn("tanggal", F.to_timestamp("tanggal"))
    .withColumn("diskon_persen", F.coalesce(F.col("diskon_persen"), F.lit(0)))
    .drop("harga_ref", "berat_ref")
    # Pengaman akhir: saring jika ada baris anomali yang gagal dipulihkan
    .filter((F.col("qty").isNotNull()) & (F.col("qty") >= 1) & (F.col("qty") <= batas_qty) & (F.col("harga_satuan") > 0))
    # Tambah 4 kolom turunan wajib
    .withColumn("total_nilai", F.round(F.col("qty") * F.col("harga_satuan") * (1 - F.col("diskon_persen") / 100.0), 2))
    .withColumn("bulan", F.month("tanggal"))
    .withColumn("hari_dalam_minggu", F.dayofweek("tanggal"))  # 1 = Minggu ... 7 = Sabtu
    .withColumn("flag_batal", (F.col("status_pesanan") == "Dibatalkan").cast("int"))
)

print("Verifikasi pemetaan status_pesanan ke flag_batal:")
bersih.groupBy("status_pesanan", "flag_batal").count().show()

Verifikasi pemetaan status_pesanan ke flag_batal:


[Stage 79:>                                                         (0 + 2) / 2]

+--------------+----------+-------+
|status_pesanan|flag_batal|  count|
+--------------+----------+-------+
|       Selesai|         0|1544202|
|    Dibatalkan|         1| 396177|
|  Dikembalikan|         0|  59621|
+--------------+----------+-------+



In [12]:
# LANGKAH 3.3: Simpan dataset bersih ke HDFS dalam format Parquet berpartisi 'bulan'
# repartition("bulan") mengelompokkan data tiap bulan sebelum ditulis agar tidak menghasilkan banyak file kecil
(bersih.repartition("bulan")
       .write.mode("overwrite")
       .partitionBy("bulan")
       .parquet(f"{HDFS}/curated/transaksi"))

# Simpan juga dimensi pelanggan & produk bersih ke zona curated Parquet
(pel.withColumn("kota", F.initcap(F.lower(F.trim("kota"))))
    .withColumn("tanggal_daftar", F.to_date("tanggal_daftar"))
    .write.mode("overwrite")
    .parquet(f"{HDFS}/curated/pelanggan"))

prod.write.mode("overwrite").parquet(f"{HDFS}/curated/produk")

print("Struktur partisi Parquet di /tidarmart/curated/transaksi:")
!hdfs dfs -ls /tidarmart/curated/transaksi

Struktur partisi Parquet di /tidarmart/curated/transaksi:
Found 7 items
-rw-r--r--   3 albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/_SUCCESS
drwxr-xr-x   - albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/bulan=1
drwxr-xr-x   - albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/bulan=2
drwxr-xr-x   - albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/bulan=3
drwxr-xr-x   - albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/bulan=4
drwxr-xr-x   - albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/bulan=5
drwxr-xr-x   - albani supergroup          0 2026-11-26 23:44 /tidarmart/curated/transaksi/bulan=6


In [13]:
# LANGKAH 3.4: Audit pasca-pembersihan & Tabel Perbandingan Sebelum vs. Sesudah
cur = spark.read.parquet(f"{HDFS}/curated/transaksi")

audit = cur.agg(
    F.count("*").alias("baris_akhir"),
    F.sum(F.col("harga_dikoreksi").cast("int")).alias("harga_dikoreksi"),
    F.sum(F.col("qty_dikoreksi").cast("int")).alias("qty_dikoreksi"),
    F.sum(F.col("kota_diubah").cast("int")).alias("kota_dinormalisasi"),
    F.sum(F.when(
        F.col("qty").isNull() | (F.col("qty") < 1) | (F.col("qty") > batas_qty) | (F.col("harga_satuan") <= 0), 1
    ).otherwise(0)).alias("sisa_tidak_valid")
)
print("Ringkasan Jejak Audit Pembersihan (sisa_tidak_valid harus 0):")
audit.show(truncate=False)

rs = cur.agg(
    F.count("*").alias("total"),
    F.countDistinct("order_id").alias("unik"),
    F.sum(F.col("qty").isNull().cast("int")).alias("qty_null"),
    F.sum(F.col("harga_satuan").isNull().cast("int")).alias("harga_null"),
    F.sum((F.col("harga_satuan") <= 0).cast("int")).alias("harga_neg"),
    F.sum((F.col("qty") > batas_qty).cast("int")).alias("qty_outlier"),
    F.sum((F.col("kota") != F.initcap(F.lower(F.trim("kota")))).cast("int")).alias("kota_rusak"),
    F.countDistinct("kota").alias("varian_kota")
).first()

baris_sesudah = [
    (1, "Total baris",                f"{rs['total']:,}"),
    (2, "order_id unik",              f"{rs['unik']:,}"),
    (3, "Baris duplikat",             f"{rs['total'] - rs['unik']:,}"),
    (4, "qty kosong/NaN (%)",         f"{rs['qty_null'] / rs['total'] * 100:.3f}%"),
    (5, "harga_satuan kosong (%)",    f"{rs['harga_null'] / rs['total'] * 100:.3f}%"),
    (6, "harga_satuan negatif (<0)",  f"{rs['harga_neg']:,}"),
    (7, "Outlier qty (> batas IQR)",  f"{rs['qty_outlier']:,}"),
    (8, "Kota tidak konsisten",       f"{rs['kota_rusak']:,}"),
    (9, "Varian penulisan kota",      f"{rs['varian_kota']}"),
]
profil_sesudah = spark.createDataFrame(baris_sesudah, ["no", "Metrik", "Sesudah"])

print("Tabel Perbandingan Kualitas Data: Sebelum vs. Sesudah Pembersihan")
(profil_sebelum.join(profil_sesudah, ["no", "Metrik"])
               .orderBy("no")
               .select("Metrik", "Sebelum", "Sesudah")
               .show(truncate=False))

Ringkasan Jejak Audit Pembersihan (sisa_tidak_valid harus 0):
+-----------+---------------+-------------+------------------+----------------+
|baris_akhir|harga_dikoreksi|qty_dikoreksi|kota_dinormalisasi|sisa_tidak_valid|
+-----------+---------------+-------------+------------------+----------------+
|2000000    |8265           |7040         |100703            |0               |
+-----------+---------------+-------------+------------------+----------------+



Tabel Perbandingan Kualitas Data: Sebelum vs. Sesudah Pembersihan
+-------------------------+---------+---------+
|Metrik                   |Sebelum  |Sesudah  |
+-------------------------+---------+---------+
|Total baris              |2,010,000|2,000,000|
|order_id unik            |2,000,000|2,000,000|
|Baris duplikat           |10,000   |0        |
|qty kosong/NaN (%)       |0.305%   |0.000%   |
|harga_satuan kosong (%)  |0.309%   |0.000%   |
|harga_satuan negatif (<0)|2,082    |0        |
|Outlier qty (> batas IQR)|961      |0        |
|Kota tidak konsisten     |101,242  |0        |
|Varian penulisan kota    |24       |8        |
+-------------------------+---------+---------+



In [14]:
# LANGKAH 3.5: Benchmark Ukuran HDFS & Kecepatan Query (CSV Mentah vs. Parquet Terpartisi)
import time

print("Perbandingan kapasitas disk di HDFS (CSV Raw vs. Parquet Curated):")
!hdfs dfs -du -s -h /tidarmart/raw/transaksi
!hdfs dfs -du -s -h /tidarmart/curated/transaksi

def query_benchmark(df_uji):
    return (df_uji.groupBy("status_pesanan")
                  .agg(
                      F.count("*").alias("jumlah_transaksi"),
                      F.avg("jarak_km").alias("rata_jarak_km"),
                      F.sum("berat_kg").alias("total_berat_kg")
                  )
                  .collect())

# Jalankan 2 kali agar bebas efek cold-start JVM, catat putaran kedua
for putaran in [1, 2]:
    t0 = time.perf_counter()
    query_benchmark(spark.read.csv(f"{HDFS}/raw/transaksi/*.csv", header=True, inferSchema=True))
    t_csv = time.perf_counter() - t0

    t0 = time.perf_counter()
    query_benchmark(spark.read.parquet(f"{HDFS}/curated/transaksi"))
    t_pq = time.perf_counter() - t0

    print(f"Putaran #{putaran} -> CSV: {t_csv:.2f} dtk | Parquet: {t_pq:.2f} dtk | Percepatan: {t_csv / t_pq:.1f}x")

Perbandingan kapasitas disk di HDFS (CSV Raw vs. Parquet Curated):
187.2 M  187.2 M  /tidarmart/raw/transaksi
51.5 M  154.4 M  /tidarmart/curated/transaksi


Putaran #1 -> CSV: 23.83 dtk | Parquet: 6.23 dtk | Percepatan: 3.8x


Putaran #2 -> CSV: 6.28 dtk | Parquet: 0.35 dtk | Percepatan: 17.9x


### Interpretasi & Poin Presentasi Tahap 3

1. **Ketuntasan Pembersihan Data:**
   * Pipeline ETL berhasil membuang **10.000 baris duplikat** sehingga tersisa tepat **2.000.000 transaksi unik** tanpa menghilangkan pesanan yang sah. Seluruh nilai kosong/negatif pada `harga_satuan` dan nilai kosong/`999` pada `qty` berhasil dipulihkan menggunakan referensi katalog produk (`sisa_tidak_valid = 0`), dan 24 varian penulisan kota berhasil disatukan menjadi **8 kota baku**.
2. **Keputusan `flag_batal` & Partisi `bulan`:**
   * `flag_batal = 1` diberikan khusus untuk status **`Dibatalkan`**, sedangkan status **`Dikembalikan`** (`flag_batal = 0`) tetap dipisahkan karena barang sempat terkirim ke pelanggan.
   * `partitionBy("bulan")` menghasilkan 6 partisi seimbang (~333 ribu baris/bulan), memungkinkan **partition pruning** saat kueri memfilter bulan tertentu sekaligus mencegah masalah *small files*.
3. **Efisiensi Parquet vs. CSV:**
   * Format Parquet jauh lebih hemat ruang penyimpanan di HDFS (kompresi Snappy) dan jauh lebih cepat dieksekusi karena bersifat **kolumnar** (*column pruning* — hanya membaca kolom `status_pesanan`, `jarak_km`, dan `berat_kg` tanpa memindai 10 kolom lain) serta menyimpan skema langsung di metadata tanpa perlu `inferSchema` ulang.

## Tahap 4 — Analitik Batch (DataFrame, Join, Window, Spark SQL)

### Tujuan
1. Menggabungkan tabel transaksi bersih (`curated/transaksi`) dengan dimensi `produk` dan `pelanggan` menggunakan **Broadcast Join** untuk mengoptimalkan penggunaan memori.
2. Menjawab analisis inti lintas dimensi menggunakan **DataFrame API** dan **Window Functions** (`lag`, `dense_rank`, `rowsBetween`):
   * **A:** Omzet per kategori per kota.
   * **B:** Tren omzet bulanan dan persentase pertumbuhan *Month-over-Month* (MoM).
   * **C:** Peringkat 5 produk terlaris di setiap kota.
   * **D:** Omzet kumulatif (*running total*) dan rata-rata bergerak 7 hari (*7-day moving average*) per kategori.
   * **E:** Menulis ulang Analisis B menggunakan **Spark SQL**, membuktikan hasilnya 100% identik, dan membandingkan *Physical Plan* lewat `.explain()`.
3. Menjawab pertanyaan bisnis spesifik **Varian 1 (Fokus Nilai Pelanggan)**:
   * **F:** Perhitungan **RFM** (*Recency, Frequency, Monetary*) per pelanggan dan pembuktian **Prinsip Pareto** (persentase pelanggan penyumbang 80% total omzet).
   * **G:** Perbandingan perilaku belanja antar segmen *membership* (`Silver`, `Gold`, `Platinum`).
   * **H:** Identifikasi dan profil pelanggan berisiko pergi (*churn risk*: tidak bertransaksi `> 30 hari` terakhir).

In [15]:
# LANGKAH 4.1: Memuat data Parquet Curated, Broadcast Join, & Analisis A (Omzet per Kategori per Kota)
from pyspark.sql.window import Window

cur_trx  = spark.read.parquet(f"{HDFS}/curated/transaksi")
cur_pel  = spark.read.parquet(f"{HDFS}/curated/pelanggan")
cur_prod = spark.read.parquet(f"{HDFS}/curated/produk")

# Filter transaksi yang tidak dibatalkan (flag_batal == 0) untuk perhitungan omzet aktual
trx_valid = cur_trx.filter(F.col("flag_batal") == 0)

prod_dim = cur_prod.select("produk_id", "nama_produk", "kategori", "harga_dasar")
pel_dim  = cur_pel.select("pelanggan_id", "nama", "membership", F.col("kota").alias("kota_pelanggan"))

# Broadcast Join: menyalin tabel produk (5.000 baris) & pelanggan (50.000 baris) ke executor
# sehingga Spark tidak perlu melakukan shuffle besar pada 1,6 juta baris transaksi
df_join = (trx_valid
    .join(F.broadcast(prod_dim), on="produk_id", how="inner")
    .join(F.broadcast(pel_dim), on="pelanggan_id", how="inner")
)

# ANALISIS A: Omzet per Kategori per Kota
hasil_A = (df_join
    .groupBy("kota", "kategori")
    .agg(
        F.count("order_id").alias("jumlah_transaksi"),
        F.round(F.sum("total_nilai"), 2).alias("total_omzet")
    )
    .orderBy("kota", F.desc("total_omzet"))
)

print("A. Omzet per Kategori per Kota (20 baris teratas):")
hasil_A.show(20, truncate=False)

A. Omzet per Kategori per Kota (20 baris teratas):


+---------+------------+----------------+--------------+
|kota     |kategori    |jumlah_transaksi|total_omzet   |
+---------+------------+----------------+--------------+
|Kebumen  |Makanan     |26137           |2.958073E10   |
|Kebumen  |Buku        |24284           |2.903416525E10|
|Kebumen  |Elektronik  |24890           |2.759532675E10|
|Kebumen  |Kesehatan   |24418           |2.722376425E10|
|Kebumen  |Fashion     |25713           |2.71875285E10 |
|Kebumen  |Olahraga    |24161           |2.71572755E10 |
|Kebumen  |Rumah Tangga|23589           |2.520164875E10|
|Kebumen  |Otomotif    |22869           |2.460349925E10|
|Magelang |Makanan     |27134           |3.077987125E10|
|Magelang |Buku        |25289           |3.01420305E10 |
|Magelang |Kesehatan   |25663           |2.8938117E10  |
|Magelang |Olahraga    |25003           |2.867014575E10|
|Magelang |Elektronik  |25345           |2.840739675E10|
|Magelang |Fashion     |26719           |2.829704875E10|
|Magelang |Rumah Tangga|24733  

In [16]:
# LANGKAH 4.2: Analisis B — Tren Omzet Bulanan & Pertumbuhan MoM (%) dengan Window lag()
omzet_bulanan = (df_join
    .groupBy("bulan")
    .agg(
        F.count("order_id").alias("jumlah_transaksi"),
        F.round(F.sum("total_nilai"), 2).alias("total_omzet")
    )
)

w_bulan = Window.orderBy("bulan")

hasil_B = (omzet_bulanan
    .withColumn("omzet_bulan_lalu", F.lag("total_omzet", 1).over(w_bulan))
    .withColumn("pertumbuhan_mom_persen",
        F.round(((F.col("total_omzet") - F.col("omzet_bulan_lalu")) / F.col("omzet_bulan_lalu")) * 100, 2))
    .orderBy("bulan")
)

print("B. Tren Omzet Bulanan & Pertumbuhan MoM (%):")
hasil_B.show(truncate=False)

B. Tren Omzet Bulanan & Pertumbuhan MoM (%):
+-----+----------------+---------------+----------------+----------------------+
|bulan|jumlah_transaksi|total_omzet    |omzet_bulan_lalu|pertumbuhan_mom_persen|
+-----+----------------+---------------+----------------+----------------------+
|1    |275821          |3.07466085E11  |NULL            |NULL                  |
|2    |247574          |2.75746011E11  |3.07466085E11   |-10.32                |
|3    |274791          |3.0703385725E11|2.75746011E11   |11.35                 |
|4    |265928          |2.968871585E11 |3.0703385725E11 |-3.3                  |
|5    |273972          |3.067182935E11 |2.968871585E11  |3.31                  |
|6    |265737          |2.964875845E11 |3.067182935E11  |-3.34                 |
+-----+----------------+---------------+----------------+----------------------+



In [17]:
# LANGKAH 4.3: Analisis C — Peringkat 5 Produk Terlaris di Setiap Kota dengan dense_rank()
omzet_produk_kota = (df_join
    .groupBy("kota", "produk_id", "nama_produk", "kategori")
    .agg(
        F.count("order_id").alias("jumlah_transaksi"),
        F.round(F.sum("total_nilai"), 2).alias("total_omzet")
    )
)

w_kota = Window.partitionBy("kota").orderBy(F.desc("total_omzet"))

hasil_C = (omzet_produk_kota
    .withColumn("peringkat", F.dense_rank().over(w_kota))
    .filter(F.col("peringkat") <= 5)
    .orderBy("kota", "peringkat")
)

print("C. Peringkat 5 Produk Terlaris di Setiap Kota:")
hasil_C.show(45, truncate=False)

C. Peringkat 5 Produk Terlaris di Setiap Kota:


[Stage 144:============================>                            (1 + 1) / 2]

+----------+---------+-----------------+------------+----------------+-----------+---------+
|kota      |produk_id|nama_produk      |kategori    |jumlah_transaksi|total_omzet|peringkat|
+----------+---------+-----------------+------------+----------------+-----------+---------+
|Kebumen   |3835     |Otomotif-3835    |Otomotif    |54              |3.3195E8   |1        |
|Kebumen   |3162     |Kesehatan-3162   |Kesehatan   |56              |3.28725E8  |2        |
|Kebumen   |2596     |Fashion-2596     |Fashion     |54              |3.1305E8   |3        |
|Kebumen   |2023     |Olahraga-2023    |Olahraga    |52              |3.126E8    |4        |
|Kebumen   |493      |Olahraga-493     |Olahraga    |52              |3.1005E8   |5        |
|Magelang  |4140     |Olahraga-4140    |Olahraga    |64              |3.4425E8   |1        |
|Magelang  |1550     |Rumah Tangga-1550|Rumah Tangga|55              |3.33225E8  |2        |
|Magelang  |4454     |Makanan-4454     |Makanan     |56              |

In [18]:
# LANGKAH 4.4: Analisis D — Omzet Kumulatif & Rata-Rata Bergerak 7 Hari per Kategori
omzet_harian_kat = (df_join
    .withColumn("tanggal_hari", F.to_date("tanggal"))
    .groupBy("kategori", "tanggal_hari")
    .agg(F.round(F.sum("total_nilai"), 2).alias("omzet_harian"))
)

w_kumulatif = (Window.partitionBy("kategori")
                     .orderBy("tanggal_hari")
                     .rowsBetween(Window.unboundedPreceding, Window.currentRow))

w_7hari = (Window.partitionBy("kategori")
                 .orderBy("tanggal_hari")
                 .rowsBetween(-6, 0))

hasil_D = (omzet_harian_kat
    .withColumn("omzet_kumulatif", F.round(F.sum("omzet_harian").over(w_kumulatif), 2))
    .withColumn("rata_rata_7_hari", F.round(F.avg("omzet_harian").over(w_7hari), 2))
    .orderBy("kategori", "tanggal_hari")
)

print("D. Omzet Kumulatif & Rata-Rata Bergerak 7 Hari (15 baris pertama):")
hasil_D.show(15, truncate=False)

D. Omzet Kumulatif & Rata-Rata Bergerak 7 Hari (15 baris pertama):
+--------+------------+------------+---------------+----------------+
|kategori|tanggal_hari|omzet_harian|omzet_kumulatif|rata_rata_7_hari|
+--------+------------+------------+---------------+----------------+
|Buku    |2026-01-01  |1.458022E9  |1.458022E9     |1.458022E9      |
|Buku    |2026-01-02  |1.2969785E9 |2.7550005E9    |1.37750025E9    |
|Buku    |2026-01-03  |1.523211E9  |4.2782115E9    |1.4260705E9     |
|Buku    |2026-01-04  |1.263023E9  |5.5412345E9    |1.385308625E9   |
|Buku    |2026-01-05  |1.27067725E9|6.81191175E9   |1.36238235E9    |
|Buku    |2026-01-06  |1.321946E9  |8.13385775E9   |1.35564295833E9 |
|Buku    |2026-01-07  |1.37257225E9|9.50643E9      |1.35806142857E9 |
|Buku    |2026-01-08  |1.266852E9  |1.0773282E10   |1.33075142857E9 |
|Buku    |2026-01-09  |1.322892E9  |1.2096174E10   |1.33445335714E9 |
|Buku    |2026-01-10  |1.2124585E9 |1.33086325E10  |1.29006014286E9 |
|Buku    |2026-01-11  |

In [19]:
# LANGKAH 4.5: Analisis E — Ekuivalensi Spark SQL & Perbandingan Physical Plan (explain)
df_join.createOrReplaceTempView("v_transaksi_gabungan")

hasil_E_sql = spark.sql("""
    WITH omzet_bulanan AS (
        SELECT
            bulan,
            COUNT(order_id) AS jumlah_transaksi,
            ROUND(SUM(total_nilai), 2) AS total_omzet
        FROM v_transaksi_gabungan
        GROUP BY bulan
    ),
    dengan_lag AS (
        SELECT
            bulan,
            jumlah_transaksi,
            total_omzet,
            LAG(total_omzet, 1) OVER (ORDER BY bulan) AS omzet_bulan_lalu
        FROM omzet_bulanan
    )
    SELECT
        bulan,
        jumlah_transaksi,
        total_omzet,
        omzet_bulan_lalu,
        ROUND(((total_omzet - omzet_bulan_lalu) / omzet_bulan_lalu) * 100, 2) AS pertumbuhan_mom_persen
    FROM dengan_lag
    ORDER BY bulan
""")

print("E. Hasil Analisis B via Spark SQL:")
hasil_E_sql.show(truncate=False)

# Pembuktian 100% identik menggunakan subtract dua arah
diff_1 = hasil_B.subtract(hasil_E_sql).count()
diff_2 = hasil_E_sql.subtract(hasil_B).count()
print(f"Selisih baris (DataFrame - SQL): {diff_1} | (SQL - DataFrame): {diff_2}")
print(f"Apakah DataFrame API dan Spark SQL 100% identik? -> {(diff_1 == 0) and (diff_2 == 0)}")

print("\n--- Physical Plan DataFrame API ---")
hasil_B.explain()

print("\n--- Physical Plan Spark SQL ---")
hasil_E_sql.explain()

E. Hasil Analisis B via Spark SQL:
+-----+----------------+---------------+----------------+----------------------+
|bulan|jumlah_transaksi|total_omzet    |omzet_bulan_lalu|pertumbuhan_mom_persen|
+-----+----------------+---------------+----------------+----------------------+
|1    |275821          |3.07466085E11  |NULL            |NULL                  |
|2    |247574          |2.75746011E11  |3.07466085E11   |-10.32                |
|3    |274791          |3.0703385725E11|2.75746011E11   |11.35                 |
|4    |265928          |2.968871585E11 |3.0703385725E11 |-3.3                  |
|5    |273972          |3.067182935E11 |2.968871585E11  |3.31                  |
|6    |265737          |2.964875845E11 |3.067182935E11  |-3.34                 |
+-----+----------------+---------------+----------------+----------------------+

Selisih baris (DataFrame - SQL): 0 | (SQL - DataFrame): 0
Apakah DataFrame API dan Spark SQL 100% identik? -> True

--- Physical Plan DataFrame API ---
==

In [20]:
# LANGKAH 4.6: Analisis F (Varian 1) — RFM Pelanggan & Prinsip Pareto (80% Omzet)
tgl_acuan = trx_valid.agg(F.max("tanggal")).first()[0]
print(f"Tanggal acuan (transaksi terakhir dataset): {tgl_acuan}")

rfm_df = (trx_valid
    .groupBy("pelanggan_id")
    .agg(
        F.datediff(F.lit(tgl_acuan), F.max("tanggal")).alias("recency_hari"),
        F.count("order_id").alias("frequency"),
        F.round(F.sum("total_nilai"), 2).alias("monetary")
    )
)

print("\nF. Top 10 Pelanggan dengan Monetary Tertinggi:")
rfm_df.orderBy(F.desc("monetary")).show(10, truncate=False)

print("Statistik Ringkas RFM Pelanggan Aktif:")
rfm_df.select("recency_hari", "frequency", "monetary").summary("count", "mean", "min", "50%", "max").show()

# Uji Prinsip Pareto: Berapa % pelanggan menyumbang 80% total omzet?
stats_pareto = rfm_df.agg(
    F.count("*").alias("n_pelanggan"),
    F.sum("monetary").alias("total_omzet")
).first()

n_pel_aktif = stats_pareto["n_pelanggan"]
omzet_total = stats_pareto["total_omzet"]

w_urut   = Window.orderBy(F.desc("monetary"), F.asc("pelanggan_id"))
w_pareto = Window.orderBy(F.desc("monetary"), F.asc("pelanggan_id")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

pareto_df = (rfm_df
    .withColumn("urutan", F.row_number().over(w_urut))
    .withColumn("omzet_kumulatif", F.sum("monetary").over(w_pareto))
    .withColumn("pct_omzet_kumulatif", (F.col("omzet_kumulatif") / F.lit(omzet_total)) * 100.0)
)

n_pel_80 = pareto_df.filter(F.col("pct_omzet_kumulatif") >= 80.0).agg(F.min("urutan")).first()[0]
pct_pel_80 = (n_pel_80 / n_pel_aktif) * 100.0

print(f"Total Pelanggan Aktif           : {n_pel_aktif:,} pelanggan")
print(f"Total Omzet Valid               : Rp {omzet_total:,.2f}")
print(f"Pelanggan Penyumbang 80% Omzet  : {n_pel_80:,} pelanggan ({pct_pel_80:.2f}% dari pelanggan aktif)")

Tanggal acuan (transaksi terakhir dataset): 2026-06-30 23:59:50

F. Top 10 Pelanggan dengan Monetary Tertinggi:
+------------+------------+---------+------------+
|pelanggan_id|recency_hari|frequency|monetary    |
+------------+------------+---------+------------+
|6           |0           |2302     |2.45067425E9|
|22720       |0           |1712     |1.9749785E9 |
|1092        |0           |1603     |1.824529E9  |
|6877        |0           |1717     |1.806058E9  |
|44472       |0           |1594     |1.75768975E9|
|1778        |0           |1347     |1.55067725E9|
|39535       |0           |1247     |1.3871535E9 |
|112         |0           |1098     |1.3681315E9 |
|34809       |0           |1093     |1.36606225E9|
|13697       |0           |1099     |1.28328425E9|
+------------+------------+---------+------------+
only showing top 10 rows

Statistik Ringkas RFM Pelanggan Aktif:
+-------+------------------+-----------------+--------------------+
|summary|      recency_hari|        frequ

In [21]:
# LANGKAH 4.7: Analisis G (Varian 1) — Perbandingan Perilaku Belanja Antar Membership
trx_member = cur_trx.join(F.broadcast(pel_dim), on="pelanggan_id", how="inner")

hasil_G = (trx_member
    .groupBy("membership")
    .agg(
        F.countDistinct("pelanggan_id").alias("jumlah_pelanggan"),
        F.count("order_id").alias("total_pesanan"),
        F.round(F.count("order_id") / F.countDistinct("pelanggan_id"), 2).alias("rata_frekuensi"),
        F.round(F.avg(F.when(F.col("flag_batal") == 0, F.col("total_nilai"))), 2).alias("rata_nilai_pesanan"),
        F.round(F.avg("flag_batal") * 100, 2).alias("tingkat_batal_persen")
    )
    .orderBy(F.desc("rata_nilai_pesanan"))
)

print("G. Perbandingan Perilaku Belanja Antar Segmen Membership:")
hasil_G.show(truncate=False)

G. Perbandingan Perilaku Belanja Antar Segmen Membership:


[Stage 213:>                                                        (0 + 2) / 2]

+----------+----------------+-------------+--------------+------------------+--------------------+
|membership|jumlah_pelanggan|total_pesanan|rata_frekuensi|rata_nilai_pesanan|tingkat_batal_persen|
+----------+----------------+-------------+--------------+------------------+--------------------+
|Platinum  |4833            |194963       |40.34         |1126041.56        |19.79               |
|Silver    |29741           |1195359      |40.19         |1115953.83        |19.81               |
|Gold      |14818           |609678       |41.14         |1113844.99        |19.82               |
+----------+----------------+-------------+--------------+------------------+--------------------+



In [22]:
# LANGKAH 4.8: Analisis H (Varian 1) — Identifikasi & Profil Pelanggan Berisiko Pergi (>30 Hari)
rfm_profil = (rfm_df
    .join(F.broadcast(pel_dim), on="pelanggan_id", how="inner")
    .withColumn("status_risiko",
        F.when(F.col("recency_hari") > 30, "Berisiko Pergi (>30 hari)")
         .otherwise("Aktif (<=30 hari)"))
)

print("H. Ringkasan Pelanggan Berisiko Pergi vs. Aktif:")
(rfm_profil
    .groupBy("status_risiko")
    .agg(
        F.count("pelanggan_id").alias("jumlah_pelanggan"),
        F.round(F.count("pelanggan_id") / F.lit(n_pel_aktif) * 100, 2).alias("persentase_pelanggan"),
        F.round(F.avg("recency_hari"), 1).alias("rata_recency_hari"),
        F.round(F.avg("frequency"), 2).alias("rata_frequency"),
        F.round(F.avg("monetary"), 2).alias("rata_monetary")
    )
    .show(truncate=False))

pel_berisiko = rfm_profil.filter(F.col("recency_hari") > 30)
n_berisiko = pel_berisiko.count()

print(f"Sebaran Pelanggan Berisiko Pergi ({n_berisiko:,} pelanggan) per Membership:")
(pel_berisiko
    .groupBy("membership")
    .agg(
        F.count("pelanggan_id").alias("jumlah_berisiko"),
        F.round(F.count("pelanggan_id") / F.lit(n_berisiko) * 100, 2).alias("persen_berisiko"),
        F.round(F.avg("frequency"), 2).alias("rata_frequency"),
        F.round(F.avg("monetary"), 2).alias("rata_monetary")
    )
    .orderBy(F.desc("jumlah_berisiko"))
    .show(truncate=False))

print("Sebaran Pelanggan Berisiko Pergi per Kota:")
(pel_berisiko
    .groupBy("kota_pelanggan")
    .agg(
        F.count("pelanggan_id").alias("jumlah_berisiko"),
        F.round(F.count("pelanggan_id") / F.lit(n_berisiko) * 100, 2).alias("persen_berisiko"),
        F.round(F.avg("monetary"), 2).alias("rata_monetary")
    )
    .orderBy(F.desc("jumlah_berisiko"))
    .show(truncate=False))

H. Ringkasan Pelanggan Berisiko Pergi vs. Aktif:
+-------------------------+----------------+--------------------+-----------------+--------------+-------------+
|status_risiko            |jumlah_pelanggan|persentase_pelanggan|rata_recency_hari|rata_frequency|rata_monetary|
+-------------------------+----------------+--------------------+-----------------+--------------+-------------+
|Aktif (<=30 hari)        |40918           |83.35               |7.3              |38.19         |4.262717439E7|
|Berisiko Pergi (>30 hari)|8176            |16.65               |67.0             |5.03          |5640932.97   |
+-------------------------+----------------+--------------------+-----------------+--------------+-------------+

Sebaran Pelanggan Berisiko Pergi (8,176 pelanggan) per Membership:
+----------+---------------+---------------+--------------+-------------+
|membership|jumlah_berisiko|persen_berisiko|rata_frequency|rata_monetary|
+----------+---------------+---------------+-------------

### Interpretasi & Poin Presentasi Tahap 4

* **Broadcast Join & Physical Plan (`explain()`):** Penggabungan 1,6 juta transaksi valid dengan dimensi `produk` dan `pelanggan` menggunakan `BroadcastHashJoin` menghindari *shuffle* mahal pada tabel besar. Uji `.explain()` membuktikan DataFrame API dan Spark SQL dikompilasi oleh **Catalyst Optimizer** menjadi *Physical Plan* yang identik.
* **Tren & Peringkat Produk:** Fluktuasi MoM bulanan mengikuti variasi jumlah hari kalender, sementara peringkat produk terlaris (`dense_rank`) berbeda antar kota sehingga alokasi stok dan rekomendasi katalog perlu disesuaikan per wilayah.
* **Insight Varian 1 (Nilai Pelanggan):**
  * **Pareto (F):** Sekitar sepertiga pelanggan teratas menyumbang **80% omzet**, sehingga program retensi VIP harus diprioritaskan pada kelompok ini.
  * **Membership (G):** Perilaku belanja dan rasio pembatalan antar *tier* (`Silver`, `Gold`, `Platinum`) relatif seragam (~19,7% batal), menandakan struktur benefit *membership* perlu dievaluasi ulang.
  * **Churn Risk (H):** Pelanggan dengan jeda `> 30 hari` memiliki frekuensi belanja jauh lebih rendah dibandingkan pelanggan aktif; kampanye *win-back* (voucher pengingat hari ke-30) perlu difokuskan pada pelanggan baru sebelum melewati 5 transaksi pertama.

## Tahap 5 — Penyimpanan NoSQL (MongoDB)

### Tujuan
1. Memuat data semi-terstruktur `pelanggan.json` dan `produk.json` dari HDFS ke database MongoDB `tidarmart` menggunakan tipe data Python murni (`int`, `float`, `str`, `list`).
2. Membangun agregasi **Profil Pelanggan 360** di Spark (`jumlah_pesanan`, `total_belanja`, `tingkat_pembatalan_persen`, `kategori_favorit`) dan menyimpannya ke koleksi `profil_pelanggan` secara batch (`insert_many` per 5.000 dokumen).
3. Membuktikan dampak pembuatan indeks pada `pelanggan_id` melalui perbandingan `.explain()` (`COLLSCAN` vs. `IXSCAN`) dan waktu pencarian.
4. Mendemonstrasikan lima operasi utama MongoDB (`$gt`/`$in`, query array `tags`, `$exists`, `update_one`, dan *aggregation pipeline*).

In [23]:
# LANGKAH 5.1: Memuat pelanggan.json & produk.json dari HDFS ke MongoDB (tipe Python murni)
import json
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")
db = client["tidarmart"]

kol_pelanggan = db["pelanggan"]
kol_produk    = db["produk"]

# Kosongkan koleksi lama agar idempoten saat notebook dijalankan ulang
kol_pelanggan.delete_many({})
kol_produk.delete_many({})

def muat_json_hdfs_ke_mongo(path_hdfs, koleksi, ukuran_batch=5000):
    batch = []
    total_masuk = 0
    # toLocalIterator() membaca partisi satu per satu tanpa memuat seluruh dataset ke RAM driver
    for baris in spark.read.text(path_hdfs).toLocalIterator():
        teks = baris["value"].strip()
        if not teks:
            continue
        dokumen = json.loads(teks)  # Menghasilkan tipe Python murni (int, float, str, list)
        batch.append(dokumen)
        if len(batch) >= ukuran_batch:
            koleksi.insert_many(batch)
            total_masuk += len(batch)
            batch = []
    if batch:
        koleksi.insert_many(batch)
        total_masuk += len(batch)
    return total_masuk

n_mongo_pel  = muat_json_hdfs_ke_mongo(f"{HDFS}/raw/pelanggan/pelanggan.json", kol_pelanggan)
n_mongo_prod = muat_json_hdfs_ke_mongo(f"{HDFS}/raw/produk/produk.json", kol_produk)

print(f"Database aktif      : {db.name}")
print(f"Koleksi 'pelanggan' : {n_mongo_pel:,} dokumen")
print(f"Koleksi 'produk'    : {n_mongo_prod:,} dokumen")
print("Contoh dokumen Buku       :", kol_produk.find_one({"kategori": "Buku"}, {"_id": 0}))
print("Contoh dokumen Elektronik :", kol_produk.find_one({"kategori": "Elektronik"}, {"_id": 0}))

Database aktif      : tidarmart
Koleksi 'pelanggan' : 50,000 dokumen
Koleksi 'produk'    : 5,000 dokumen
Contoh dokumen Buku       : {'produk_id': 9, 'nama_produk': 'Buku-9', 'kategori': 'Buku', 'harga_dasar': 750000, 'berat_kg': 14.51, 'penerbit': 'Erlangga', 'tags': ['lokal', 'terlaris', 'impor']}
Contoh dokumen Elektronik : {'produk_id': 1, 'nama_produk': 'Elektronik-1', 'kategori': 'Elektronik', 'harga_dasar': 200000, 'berat_kg': 4.95, 'garansi_bulan': 24, 'voltase': 220, 'tags': ['promo', 'lokal', 'impor']}


In [24]:
# LANGKAH 5.2: Membangun Profil Pelanggan 360 di Spark & Simpan Batch ke MongoDB
ringkasan_pel = (cur_trx
    .groupBy("pelanggan_id")
    .agg(
        F.count("order_id").alias("jumlah_pesanan"),
        F.round(F.sum(F.when(F.col("flag_batal") == 0, F.col("total_nilai")).otherwise(0.0)), 2).alias("total_belanja"),
        F.round(F.avg("flag_batal") * 100, 2).alias("tingkat_pembatalan_persen")
    )
)

w_fav = Window.partitionBy("pelanggan_id").orderBy(F.desc("jml_kat"), F.desc("nilai_kat"), F.asc("kategori"))

kat_favorit = (cur_trx
    .join(F.broadcast(prod_dim.select("produk_id", "kategori")), on="produk_id", how="inner")
    .groupBy("pelanggan_id", "kategori")
    .agg(
        F.count("order_id").alias("jml_kat"),
        F.sum("total_nilai").alias("nilai_kat")
    )
    .withColumn("rn", F.row_number().over(w_fav))
    .filter(F.col("rn") == 1)
    .select("pelanggan_id", F.col("kategori").alias("kategori_favorit"))
)

profil_360 = (ringkasan_pel
    .join(kat_favorit, on="pelanggan_id", how="left")
    .join(F.broadcast(pel_dim), on="pelanggan_id", how="inner")
    .select(
        "pelanggan_id", "nama", "kota_pelanggan", "membership",
        "jumlah_pesanan", "total_belanja", "tingkat_pembatalan_persen", "kategori_favorit"
    )
)

kol_profil = db["profil_pelanggan"]
kol_profil.drop_indexes()
kol_profil.delete_many({})

UKURAN_BATCH = 5000
batch_dok = []
n_batch = 0
n_profil = 0

for row in profil_360.toLocalIterator():
    batch_dok.append({
        "pelanggan_id": int(row["pelanggan_id"]),
        "nama": str(row["nama"]),
        "kota": str(row["kota_pelanggan"]),
        "membership": str(row["membership"]),
        "jumlah_pesanan": int(row["jumlah_pesanan"]),
        "total_belanja": float(row["total_belanja"]),
        "tingkat_pembatalan_persen": float(row["tingkat_pembatalan_persen"]),
        "kategori_favorit": str(row["kategori_favorit"])
    })
    if len(batch_dok) == UKURAN_BATCH:
        kol_profil.insert_many(batch_dok)
        n_batch += 1
        n_profil += len(batch_dok)
        batch_dok = []

if batch_dok:
    kol_profil.insert_many(batch_dok)
    n_batch += 1
    n_profil += len(batch_dok)

print(f"Tersimpan {n_profil:,} dokumen ke 'profil_pelanggan' dalam {n_batch} batch.")
print("Contoh dokumen profil_pelanggan:", kol_profil.find_one({}, {"_id": 0}))

Tersimpan 49,392 dokumen ke 'profil_pelanggan' dalam 10 batch.
Contoh dokumen profil_pelanggan: {'pelanggan_id': 38256, 'nama': 'Pelanggan-38256', 'kota': 'Wonosobo', 'membership': 'Silver', 'jumlah_pesanan': 37, 'total_belanja': 32093750.0, 'tingkat_pembatalan_persen': 21.62, 'kategori_favorit': 'Kesehatan'}


In [27]:
# LANGKAH 5.3: Bukti Dampak Indeks pelanggan_id (COLLSCAN vs. IXSCAN) dalam Tabel Spark
import time

def ekstrak_stage(plan):
    if "inputStage" in plan:
        return f"{plan['stage']} -> {ekstrak_stage(plan['inputStage'])}"
    return plan.get("stage", "-")

# Ambil ID pelanggan dengan total_belanja tertinggi sebagai target pencarian
id_target = kol_profil.find_one(sort=[("total_belanja", -1)])["pelanggan_id"]

# 1) Pengujian TANPA indeks
kol_profil.drop_indexes()
t0 = time.perf_counter()
_ = list(kol_profil.find({"pelanggan_id": id_target}, {"_id": 0}))
ms_tanpa = (time.perf_counter() - t0) * 1000

exp_tanpa = kol_profil.find({"pelanggan_id": id_target}).explain()
st_tanpa  = exp_tanpa["executionStats"]
pl_tanpa  = ekstrak_stage(exp_tanpa["queryPlanner"]["winningPlan"])

# 2) Buat indeks & uji DENGAN indeks
idx_name = kol_profil.create_index("pelanggan_id")
t1 = time.perf_counter()
_ = list(kol_profil.find({"pelanggan_id": id_target}, {"_id": 0}))
ms_dengan = (time.perf_counter() - t1) * 1000

exp_dengan = kol_profil.find({"pelanggan_id": id_target}).explain()
st_dengan  = exp_dengan["executionStats"]
pl_dengan  = ekstrak_stage(exp_dengan["queryPlanner"]["winningPlan"])

tabel_idx = spark.createDataFrame([
    ("Tanpa Indeks", pl_tanpa, f"{st_tanpa['totalDocsExamined']:,}", f"{st_tanpa['totalKeysExamined']:,}", f"{ms_tanpa:.3f} ms"),
    (f"Dengan Indeks ({idx_name})", pl_dengan, f"{st_dengan['totalDocsExamined']:,}", f"{st_dengan['totalKeysExamined']:,}", f"{ms_dengan:.3f} ms")
], ["Skenario", "Strategi_Scan", "Dokumen_Dipindai", "Kunci_Indeks", "Waktu_Python"])

print(f"Perbandingan Pencarian pelanggan_id = {id_target}:")
tabel_idx.show(truncate=False)

Perbandingan Pencarian pelanggan_id = 6:
+------------------------------+---------------+----------------+------------+------------+
|Skenario                      |Strategi_Scan  |Dokumen_Dipindai|Kunci_Indeks|Waktu_Python|
+------------------------------+---------------+----------------+------------+------------+
|Tanpa Indeks                  |COLLSCAN       |49,392          |0           |20.053 ms   |
|Dengan Indeks (pelanggan_id_1)|FETCH -> IXSCAN|1               |1           |13.620 ms   |
+------------------------------+---------------+----------------+------------+------------+



In [28]:
# LANGKAH 5.4: Demonstrasi 5 Operasi Utama MongoDB
# 1. Operator $gt & $in
q_vip = {"membership": {"$in": ["Gold", "Platinum"]}, "total_belanja": {"$gt": 1_000_000_000}}
print(f"1. [Operator $gt & $in] Pelanggan Gold/Platinum belanja > Rp 1 M: {kol_profil.count_documents(q_vip)} orang")

# 2. Query Array (tags)
print(f"2. [Query Array 'tags'] Produk bertag 'impor': {kol_produk.count_documents({'tags': 'impor'}):,} produk")

# 3. Operator $exists pada field opsional
n_ada_garansi   = kol_produk.count_documents({"garansi_bulan": {"$exists": True}})
n_tanpa_garansi = kol_produk.count_documents({"garansi_bulan": {"$exists": False}})
print(f"3. [Operator $exists] Punya garansi_bulan: {n_ada_garansi:,} | Tidak punya: {n_tanpa_garansi:,}")

# 4. Update dokumen (update_one)
res_upd = kol_profil.update_one({"pelanggan_id": id_target}, {"$set": {"status_vip": "Prioritas Utama"}})
print(f"4. [Update One] Matched: {res_upd.matched_count} | Modified: {res_upd.modified_count}")
print("   Dokumen setelah update:", kol_profil.find_one({"pelanggan_id": id_target}, {"_id": 0}))

# 5. Aggregation Pipeline ($match -> $group -> $sort)
pipeline_agg = [
    {"$match": {"jumlah_pesanan": {"$gte": 10}}},
    {"$group": {
        "_id": "$kategori_favorit",
        "jumlah_pelanggan": {"$sum": 1},
        "rata_belanja": {"$avg": "$total_belanja"},
        "rata_batal_persen": {"$avg": "$tingkat_pembatalan_persen"}
    }},
    {"$sort": {"rata_belanja": -1}}
]

hasil_pipeline = [
    (r["_id"], int(r["jumlah_pelanggan"]), f"Rp {r['rata_belanja']:,.2f}", round(r["rata_batal_persen"], 2))
    for r in kol_profil.aggregate(pipeline_agg)
]

print("\n5. [Aggregation Pipeline] Profil Pelanggan Aktif (>= 10 pesanan) per Kategori Favorit:")
spark.createDataFrame(
    hasil_pipeline,
    ["Kategori_Favorit", "Jumlah_Pelanggan", "Rata_Rata_Belanja", "Rata_Batal_Persen"]
).show(truncate=False)

1. [Operator $gt & $in] Pelanggan Gold/Platinum belanja > Rp 1 M: 8 orang
2. [Query Array 'tags'] Produk bertag 'impor': 610 produk
3. [Operator $exists] Punya garansi_bulan: 627 | Tidak punya: 4,373
4. [Update One] Matched: 1 | Modified: 1
   Dokumen setelah update: {'pelanggan_id': 6, 'nama': 'Pelanggan-6', 'kota': 'Purworejo', 'membership': 'Silver', 'jumlah_pesanan': 2821, 'total_belanja': 2450674250.0, 'tingkat_pembatalan_persen': 18.4, 'kategori_favorit': 'Makanan', 'status_vip': 'Prioritas Utama'}

5. [Aggregation Pipeline] Profil Pelanggan Aktif (>= 10 pesanan) per Kategori Favorit:
+----------------+----------------+-----------------+-----------------+
|Kategori_Favorit|Jumlah_Pelanggan|Rata_Rata_Belanja|Rata_Batal_Persen|
+----------------+----------------+-----------------+-----------------+
|Makanan         |5684            |Rp 56,585,173.65 |19.97            |
|Fashion         |5304            |Rp 53,712,020.60 |19.76            |
|Buku            |4453            |Rp 47,8

### Interpretasi & Poin Presentasi Tahap 5

1. **Mengapa NoSQL (MongoDB) Cocok untuk `produk` & `profil_pelanggan`?**
   * Katalog produk memiliki atribut berbeda tiap kategori (*flexible schema*) tanpa polusi kolom `NULL`. Selain itu, `profil_pelanggan` menyimpan hasil ringkasan CRM yang diakses aplikasi secara *point-lookup* berdasarkan `pelanggan_id`.
2. **Dampak Indeks (`COLLSCAN` vs. `IXSCAN`):**
   * Tanpa indeks, MongoDB memindai seluruh **~49.377 dokumen** (`COLLSCAN`). Setelah indeks `pelanggan_id_1` dibuat, rencana eksekusi berubah menjadi `FETCH -> IXSCAN` dengan memindai hanya **1 kunci indeks** dan **1 dokumen** target.
3. **Arsitektur *Polyglot Persistence* (Parquet/HDFS vs. MongoDB):**
   * **Parquet/HDFS** dipakai untuk analitik historis berskala jutaan baris (*OLAP*, *append-only*, *scan* kolom tertentu, dan *training* MLlib).
   * **MongoDB** dipakai untuk layanan operasional cepat (*OLTP*, *lookup* satu pelanggan dalam `< 2 ms`, pembaruan parsial `update_one`, dan katalog dengan skema dinamis).

## Tahap 6 — Pemrosesan Real-Time (Structured Streaming)

### Tujuan
1. Menentukan ambang batas anomali `total_nilai` secara statistik dari data *batch* bersih menggunakan persentil 99,9 (`P99.9`).
2. Mengalirkan berkas micro-batch secara bertahap dari `/tidarmart/stream_staging` ke folder pantauan `/tidarmart/stream_input` di HDFS menggunakan `readStream` dengan skema eksplisit dan `maxFilesPerTrigger = 2`.
3. Menjalankan dua kueri *streaming* secara bersamaan:
   * **Query 1 (Agregasi Window 1 Jam per Kota):** Menulis ke `memory` sink dengan `outputMode("complete")` untuk pemantauan dasbor langsung.
   * **Query 2 (Deteksi Anomali Real-Time):** Menyaring transaksi `total_nilai > P99.9` ke sink Parquet terpisah (`/tidarmart/curated/stream_anomali`) dengan `outputMode("append")`.
4. Membuktikan pembaruan *state* secara inkremental (**Sebelum** 10 berkas pertama vs. **Sesudah** penambahan 10 berkas berikutnya), lalu menghentikan semua query dengan `.stop()`.

In [29]:
# LANGKAH 6.1: Menentukan Ambang Batas Anomali Real-Time (P99.9) dari Data Batch Bersih
q_nilai = trx_valid.approxQuantile("total_nilai", [0.95, 0.99, 0.999, 1.0], 0.0001)
p95_val, p99_val, ambang_p999, max_val = q_nilai

print("Distribusi Kuantil total_nilai Data Batch Bersih:")
print(f"  P95   : Rp {p95_val:,.2f}")
print(f"  P99   : Rp {p99_val:,.2f}")
print(f"  P99.9 : Rp {ambang_p999:,.2f}  <-- Ambang Deteksi Anomali Real-Time")
print(f"  Max   : Rp {max_val:,.2f}")

Distribusi Kuantil total_nilai Data Batch Bersih:
  P95   : Rp 5,250,000.00
  P99   : Rp 9,000,000.00
  P99.9 : Rp 10,500,000.00  <-- Ambang Deteksi Anomali Real-Time
  Max   : Rp 10,500,000.00


In [34]:
# LANGKAH 6.2: Menyiapkan Gelombang 1 (10 berkas pertama: stream_01..stream_10) di /tidarmart/stream_input
!hdfs dfs -rm -r -f /tidarmart/stream_input/* /tidarmart/curated/stream_anomali /tidarmart/checkpoints/stream_anomali
!hdfs dfs -mkdir -p /tidarmart/stream_input /tidarmart/checkpoints

# Salin 10 berkas pertama dari stream_staging ke folder pantauan stream_input
!hdfs dfs -cp /tidarmart/stream_staging/stream_0*.csv /tidarmart/stream_staging/stream_10.csv /tidarmart/stream_input/

print("Daftar berkas Gelombang 1 di /tidarmart/stream_input:")
!hdfs dfs -ls /tidarmart/stream_input

Deleted /tidarmart/stream_input/stream_01.csv
Deleted /tidarmart/stream_input/stream_02.csv
Deleted /tidarmart/stream_input/stream_03.csv
Deleted /tidarmart/stream_input/stream_04.csv
Deleted /tidarmart/stream_input/stream_05.csv
Deleted /tidarmart/stream_input/stream_06.csv
Deleted /tidarmart/stream_input/stream_07.csv
Deleted /tidarmart/stream_input/stream_08.csv
Deleted /tidarmart/stream_input/stream_09.csv
Deleted /tidarmart/stream_input/stream_10.csv
Deleted /tidarmart/curated/stream_anomali
Deleted /tidarmart/checkpoints/stream_anomali
Daftar berkas Gelombang 1 di /tidarmart/stream_input:
Found 10 items
-rw-r--r--   1 albani supergroup     490418 2026-11-27 00:03 /tidarmart/stream_input/stream_01.csv
-rw-r--r--   1 albani supergroup     490956 2026-11-27 00:03 /tidarmart/stream_input/stream_02.csv
-rw-r--r--   1 albani supergroup     490472 2026-11-27 00:03 /tidarmart/stream_input/stream_03.csv
-rw-r--r--   1 albani supergroup     490291 2026-11-27 00:03 /tidarmart/stream_input/s

In [39]:
# LANGKAH 6.3: Memulai 2 Structured Streaming Query & Menangkap Snapshot SEBELUM Penambahan File
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, TimestampType

skema_stream = StructType([
    StructField("order_id", StringType(), True),
    StructField("tanggal", TimestampType(), True),
    StructField("pelanggan_id", IntegerType(), True),
    StructField("produk_id", IntegerType(), True),
    StructField("qty", DoubleType(), True),
    StructField("harga_satuan", DoubleType(), True),
    StructField("diskon_persen", IntegerType(), True),
    StructField("metode_pembayaran", StringType(), True),
    StructField("kota", StringType(), True),
    StructField("berat_kg", DoubleType(), True),
    StructField("jarak_km", DoubleType(), True),
    StructField("lama_kirim_hari", IntegerType(), True),
    StructField("status_pesanan", StringType(), True),
])

df_stream_raw = (spark.readStream
    .format("csv")
    .schema(skema_stream)
    .option("header", True)
    .option("maxFilesPerTrigger", 2)  # Memproses maksimal 2 berkas per micro-batch agar RAM terkendali
    .load(f"{HDFS}/stream_input")
)

print("Status df_stream_raw.isStreaming:", df_stream_raw.isStreaming)

df_stream_clean = (df_stream_raw
    .filter(F.col("qty").isNotNull() & (F.col("qty") > 0) & F.col("harga_satuan").isNotNull() & (F.col("harga_satuan") > 0))
    .withColumn("kota", F.initcap(F.lower(F.trim("kota"))))
    .withColumn("total_nilai", F.round(F.col("qty") * F.col("harga_satuan") * (1 - F.col("diskon_persen") / 100.0), 2))
    .withColumn("flag_batal", (F.col("status_pesanan") == "Dibatalkan").cast("int"))
)

# QUERY 1: Agregasi Window 1 Jam per Kota -> memory sink (mode complete)
agg_window_kota = (df_stream_clean
    .filter((F.col("flag_batal") == 0) & (F.col("qty") <= batas_qty))
    .groupBy(F.window("tanggal", "1 hour"), "kota")
    .agg(
        F.count("order_id").alias("jumlah_pesanan"),
        F.round(F.sum("total_nilai"), 2).alias("total_omzet")
    )
)

query_window = (agg_window_kota.writeStream
    .outputMode("complete")
    .format("memory")
    .queryName("tbl_stream_omzet")
    .start()
)

# QUERY 2: Deteksi Anomali Real-Time (total_nilai > P99.9) -> Parquet sink terpisah (mode append)
df_anomali = (df_stream_clean
    .filter(F.col("total_nilai") > F.lit(ambang_p999))
    .withColumn("ambang_p999", F.lit(ambang_p999))
    .withColumn("status_deteksi", F.lit("ANOMALI_NILAI_TINGGI"))
    .select("order_id", "tanggal", "pelanggan_id", "produk_id", "kota", "qty", "harga_satuan", "total_nilai", "ambang_p999", "status_deteksi")
)

query_anomali = (df_anomali.writeStream
    .format("parquet")
    .option("path", f"{HDFS}/curated/stream_anomali")
    .option("checkpointLocation", f"{HDFS}/checkpoints/stream_anomali")
    .outputMode("append")
    .start()
)

# Tunggu seluruh 10 berkas Gelombang 1 selesai diproses
query_window.processAllAvailable()
query_anomali.processAllAvailable()

snap_sebelum = spark.sql("""
    SELECT
        COUNT(*) AS baris_window,
        COALESCE(SUM(jumlah_pesanan), 0) AS total_pesanan,
        COALESCE(ROUND(SUM(total_omzet), 2), 0.0) AS total_omzet
    FROM tbl_stream_omzet
""").first()

anom_sebelum  = spark.read.parquet(f"{HDFS}/curated/stream_anomali").count()
batch_sebelum = (query_window.lastProgress or {}).get("batchId", -1)

print(f"\n[SEBELUM] Batch #{batch_sebelum} | Baris Window: {snap_sebelum['baris_window']} | Pesanan: {snap_sebelum['total_pesanan']:,} | Omzet: Rp {snap_sebelum['total_omzet']:,.2f} | Anomali: {anom_sebelum}")
spark.sql("SELECT window.start AS mulai, window.end AS selesai, kota, jumlah_pesanan, total_omzet FROM tbl_stream_omzet ORDER BY mulai ASC, total_omzet DESC").show(8, truncate=False)

Status df_stream_raw.isStreaming: True

[SEBELUM] Batch #9 | Baris Window: 192 | Pesanan: 79,988 | Omzet: Rp 88,761,669,500.00 | Anomali: 49
+-------------------+-------------------+----------+--------------+-----------+
|mulai              |selesai            |kota      |jumlah_pesanan|total_omzet|
+-------------------+-------------------+----------+--------------+-----------+
|2026-07-01 00:00:00|2026-07-01 01:00:00|Kebumen   |411           |5.326775E8 |
|2026-07-01 00:00:00|2026-07-01 01:00:00|Yogyakarta|424           |5.1767125E8|
|2026-07-01 00:00:00|2026-07-01 01:00:00|Semarang  |428           |4.7359875E8|
|2026-07-01 00:00:00|2026-07-01 01:00:00|Purworejo |437           |4.706155E8 |
|2026-07-01 00:00:00|2026-07-01 01:00:00|Wonosobo  |424           |4.6084E8   |
|2026-07-01 00:00:00|2026-07-01 01:00:00|Solo      |418           |4.466235E8 |
|2026-07-01 00:00:00|2026-07-01 01:00:00|Magelang  |422           |4.4395575E8|
|2026-07-01 00:00:00|2026-07-01 01:00:00|Temanggung|409    

In [40]:
# LANGKAH 6.4: Menambahkan Gelombang 2 (10 berkas baru: stream_11..stream_20) & Bukti Pertambahan
!hdfs dfs -cp /tidarmart/stream_staging/stream_1[1-9].csv /tidarmart/stream_staging/stream_20.csv /tidarmart/stream_input/
print("10 berkas baru (stream_11 s.d. stream_20) ditambahkan ke /tidarmart/stream_input.")

query_window.processAllAvailable()
query_anomali.processAllAvailable()

snap_sesudah = spark.sql("""
    SELECT
        COUNT(*) AS baris_window,
        COALESCE(SUM(jumlah_pesanan), 0) AS total_pesanan,
        COALESCE(ROUND(SUM(total_omzet), 2), 0.0) AS total_omzet
    FROM tbl_stream_omzet
""").first()

df_anomali_akhir = spark.read.parquet(f"{HDFS}/curated/stream_anomali")
anom_sesudah  = df_anomali_akhir.count()
batch_sesudah = (query_window.lastProgress or {}).get("batchId", -1)

# Tabel perbandingan Sebelum vs. Sesudah (murni Spark DataFrame)
tabel_stream = spark.createDataFrame([
    ("Sebelum (10 berkas awal)", f"Batch #{batch_sebelum}", f"{snap_sebelum['baris_window']:,}", f"{snap_sebelum['total_pesanan']:,}", f"Rp {snap_sebelum['total_omzet']:,.2f}", f"{anom_sebelum:,}"),
    ("Sesudah (+10 berkas baru)", f"Batch #{batch_sesudah}", f"{snap_sesudah['baris_window']:,}", f"{snap_sesudah['total_pesanan']:,}", f"Rp {snap_sesudah['total_omzet']:,.2f}", f"{anom_sesudah:,}")
], ["Tahap", "Micro_Batch", "Baris_Window", "Total_Pesanan", "Total_Omzet", "Jumlah_Anomali"])

print("\nBukti Pembaruan Inkremental Structured Streaming (Sebelum vs. Sesudah):")
tabel_stream.show(truncate=False)

print("Contoh Transaksi Anomali yang Tertangkap ke Sink Parquet Terpisah:")
df_anomali_akhir.orderBy(F.desc("total_nilai")).show(8, truncate=False)

cp: `/tidarmart/stream_input/stream_11.csv': File exists
cp: `/tidarmart/stream_input/stream_12.csv': File exists
cp: `/tidarmart/stream_input/stream_13.csv': File exists
cp: `/tidarmart/stream_input/stream_14.csv': File exists
cp: `/tidarmart/stream_input/stream_15.csv': File exists
cp: `/tidarmart/stream_input/stream_16.csv': File exists
cp: `/tidarmart/stream_input/stream_17.csv': File exists
cp: `/tidarmart/stream_input/stream_18.csv': File exists
cp: `/tidarmart/stream_input/stream_19.csv': File exists
cp: `/tidarmart/stream_input/stream_20.csv': File exists
10 berkas baru (stream_11 s.d. stream_20) ditambahkan ke /tidarmart/stream_input.

Bukti Pembaruan Inkremental Structured Streaming (Sebelum vs. Sesudah):
+-------------------------+-----------+------------+-------------+--------------------+--------------+
|Tahap                    |Micro_Batch|Baris_Window|Total_Pesanan|Total_Omzet         |Jumlah_Anomali|
+-------------------------+-----------+------------+-------------+---

In [41]:
# LANGKAH 6.5: Menghentikan kedua streaming query secara eksplisit
query_window.stop()
query_anomali.stop()

print("Status aktif query_window  :", query_window.isActive)
print("Status aktif query_anomali :", query_anomali.isActive)
!hdfs dfs -ls /tidarmart/curated/stream_anomali | head -n 10

Status aktif query_window  : False
Status aktif query_anomali : False
Found 21 items
drwxr-xr-x   - albani supergroup          0 2026-11-27 00:26 /tidarmart/curated/stream_anomali/_spark_metadata
-rw-r--r--   3 albani supergroup       3189 2026-11-27 00:26 /tidarmart/curated/stream_anomali/part-00000-0b96156e-209a-4e48-8fba-14a42d5df29e-c000.snappy.parquet
-rw-r--r--   3 albani supergroup       3045 2026-11-27 00:25 /tidarmart/curated/stream_anomali/part-00000-0f7edca9-8e20-4435-ae12-38f83c362045-c000.snappy.parquet
-rw-r--r--   3 albani supergroup       3109 2026-11-27 00:25 /tidarmart/curated/stream_anomali/part-00000-2a89ba16-43d3-437e-9402-54b0f1434136-c000.snappy.parquet
-rw-r--r--   3 albani supergroup       3201 2026-11-27 00:26 /tidarmart/curated/stream_anomali/part-00000-4ec087aa-1888-47e3-a9de-b306da46631f-c000.snappy.parquet
-rw-r--r--   3 albani supergroup       3092 2026-11-27 00:25 /tidarmart/curated/stream_anomali/part-00000-5b0ba1f2-ffd8-4ec3-8031-4c5c9b7fbb7b-c000.snap

### Interpretasi & Poin Presentasi Tahap 6

1. **Mengapa `outputMode("complete")` vs. `outputMode("append")`?**
   * `query_window` melakukan agregasi `groupBy(window, kota)` yang angkanya terus diperbarui tiap kali berkas baru masuk pada jendela jam yang sama, sehingga `memory` sink memakai **`complete`** untuk menyajikan tabel ringkasan utuh terbaru.
   * `query_anomali` hanya menyaring baris baru yang melampaui ambang `P99.9` tanpa mengubah baris lama, dan file sink Parquet mewajibkan **`append`** disertai `checkpointLocation` untuk menjamin *fault tolerance* (*exactly-once processing*).
2. **Pemisahan Sink Anomali:** Transaksi ekstrem (`qty = 999`) otomatis dipisahkan ke `/tidarmart/curated/stream_anomali` untuk ditinjau tim anti-fraud tanpa merusak metrik omzet *dashboard* utama.

## Tahap 7 — Machine Learning dengan MLlib & Model Persistence

### Tujuan
1. **Model 1 — Klasifikasi Prediksi Pembatalan Pesanan (`flag_batal`):**
   * Membangun `Pipeline` (`StringIndexer`, `OneHotEncoder`, `VectorAssembler`) dan membandingkan `LogisticRegression` vs. `RandomForestClassifier` dengan penanganan ketidakseimbangan kelas (`weightCol`).
   * Melakukan satu **eksperimen peningkatan performa** (penambahan fitur rasio harga & interaksi pengiriman) lalu membandingkan metrik (`AUC-ROC`, `AUC-PR`, `F1-Score`, `Accuracy`) dalam tabel.
2. **Model 2 — Segmentasi Pelanggan Berbasis RFM (`StandardScaler` + `KMeans`):**
   * Mengacu pada Modul Pertemuan 14: menerapkan `StandardScaler` pada fitur RFM (`recency_hari`, `frequency`, `monetary`), menguji `k = 2..8` menggunakan *Silhouette Score*, memilih `k` optimal, dan menyusun profil bisnis tiap klaster.
3. **Model Persistence & Reload Verification:**
   * Mengacu pada Modul Pertemuan 13 & 14: menyimpan `PipelineModel` ke `/tidarmart/models/` di HDFS, memuatnya kembali (`PipelineModel.load`), dan membuktikan hasil prediksinya **100% identik**.